# Tarea 2 Parte A
Integrantes:
- Carlos Raúl Sánchez Figueroa
- Ulises Omar Montes Correa
- Diego Córdoba Gómez
- Guillermo Collado

#Inciso 1)

In [0]:
import pyspark.sql.functions as F

In [0]:
df_silver = spark.table("dev.ciencias_data.silver_sessions")
display(df_silver.limit(5))

##Tamaño y estructura de los datos

In [0]:
n_rows = df_silver.count()
n_cols = len(df_silver.columns)

print(f"Número de registros: {n_rows}")
print(f"Número de columnas: {n_cols}")

df_silver.printSchema()

###Interpretación

La tabla silver contiene 29,556 registros con múltiples variables que describen el tráfico de red.
Los tipos de datos son consistentes con la naturaleza de cada variable, permitiendo su análisis posterior.

##Duplicados

In [0]:
total = df_silver.count()
sin_dup = df_silver.dropDuplicates().count()

duplicados = total - sin_dup

print(f"Duplicados: {duplicados}")

In [0]:
total = df_silver.count()
sin_dup = df_silver.select("community_id").dropDuplicates().count()

duplicados = total - sin_dup

print(f"Duplicados: {duplicados}")


###Interpretación

No se identificaron registros completos duplicados en el dataset, aunque si utlizamos el registro unico community id podemos encontrar 3677 registros duplicados. Para no complicarnos de más y observar un por qué, hemos decidido eliminarlos porteriomente al analisis.

##Análisis de valores nulos

In [0]:
df_nulls = df_silver.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_silver.columns
])

df_nulls_long = df_nulls.select(
    F.explode(
        F.array([
            F.struct(F.lit(c).alias("columna"), F.col(c).alias("nulos"))
            for c in df_nulls.columns
        ])
    ).alias("tmp")
).select("tmp.*")

display(df_nulls_long)

Databricks visualization. Run in Databricks to view.

###Interpretación

Se identificó una alta proporción de valores nulos en variables como src_asn, dst_asn, src_geo, dst_geo e init_rtt, y otras variables como community_id.
Esto indica que la información de geolocalización, sistema autónomo y latencia no está disponible para una gran parte de las sesiones.
Por lo tanto, estas variables no se consideran críticas para el análisis posterior.

##Análisis de valores cero

In [0]:
df_zeros = df_silver.select(
    F.count(F.when(F.col("tot_bytes") == 0, True)).alias("tot_bytes"),
    F.count(F.when(F.col("tot_packets") == 0, True)).alias("tot_packets"),
    F.count(F.when(F.col("tot_data_bytes") == 0, True)).alias("tot_data_bytes")
)

df_zeros_long = df_zeros.select(
    F.explode(
        F.array([
            F.struct(F.lit("tot_bytes").alias("variable"), F.col("tot_bytes").alias("zeros")),
            F.struct(F.lit("tot_packets").alias("variable"), F.col("tot_packets").alias("zeros")),
            F.struct(F.lit("tot_data_bytes").alias("variable"), F.col("tot_data_bytes").alias("zeros"))
        ])
    ).alias("tmp")
).select("tmp.*")

display(df_zeros_long)

Databricks visualization. Run in Databricks to view.

###Interpretación

Se observaron 5430 registros con valor cero en tot_data_bytes, lo cual puede representar sesiones sin transferencia efectiva de datos.
Este comportamiento es consistente con tráfico de red y no se considera un error.

##Validación de consistencia

In [0]:
df_consistency = df_silver.withColumn(
    "diff_bytes",
    F.col("tot_bytes") - (F.col("src_bytes") + F.col("dst_bytes"))
)

display(df_consistency.select("diff_bytes").limit(10))
print('¿Cuántos hay distintos de cero?:', df_consistency.select("diff_bytes").filter(F.col("diff_bytes") != 0).count())

Databricks visualization. Run in Databricks to view.

###Interpretación

La diferencia entre tot_bytes y la suma de src_bytes y dst_bytes es igual a cero en todos los registros, lo cual confirma que los datos son completamente consistentes.

##Estadísticas descriptivas

In [0]:
cols = ["tot_bytes", "tot_packets", "tot_data_bytes"]
stats = (
    df_silver
    .select(
        *[F.col(c) for c in cols]
    )
    .summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max")
)

display(stats)


Databricks visualization. Run in Databricks to view.

###Interpretación

Se observa una alta dispersión en las variables numéricas, especialmente en tot_bytes, donde existen valores extremos significativamente mayores al promedio.
Esto indica la presencia de sesiones de alto volumen, lo cual es esperado en tráfico de red.

##Conclusión general
Se realizó un análisis de calidad de los datos en la capa silver, evaluando duplicados, valores nulos, valores cero, consistencia y estadísticas descriptivas.
No se encontraron registros duplicados, lo cual garantiza la integridad del dataset.
Se identificó una alta proporción de valores nulos en variables relacionadas con geolocalización y sistema autónomo, por lo que su uso en el análisis será limitado.
Las variables principales de tráfico (tot_bytes, tot_packets, tot_data_bytes) presentan buena calidad y consistencia.
Se detectaron valores cero en tot_data_bytes, los cuales son coherentes con la naturaleza del tráfico de red.
Asimismo, se observó una alta dispersión en las variables numéricas, con presencia de valores extremos esperados en este tipo de datos.
Finalmente, se confirmó la consistencia total de los datos, lo que permite continuar con confianza hacia la etapa de Feature Engineering y modelado.

#Inciso 2)


In [0]:
import pyspark.sql.functions as F

df_silver = spark.table("dev.ciencias_data.silver_sessions")

In [0]:
print(df_silver.count())
df_features = df_silver

In [0]:
df_features = (
    df_features
    .withColumn(
        "session_duration",
        (F.col("last_packet").cast("long") - F.col("first_packet").cast("long")).cast("long")
    )
    .withColumn(
        "bytes_per_second",
        F.when(F.col("session_duration") > 0,
               (F.col("tot_bytes") / F.col("session_duration")).cast("double"))
         .otherwise(F.lit(0.0))
    )
    .withColumn(
        "avg_packet_size",
        F.when(F.col("tot_packets") > 0,
               (F.col("tot_bytes") / F.col("tot_packets")).cast("double"))
         .otherwise(F.lit(0.0))
    )
    .withColumn(
        "bytes_ratio_src_dst",
        F.when(F.col("dst_bytes") > 0,
               (F.col("src_bytes") / F.col("dst_bytes")).cast("double"))
         .otherwise(F.lit(0.0))
    )
    .withColumn(
        "packets_ratio_src_dst",
        F.when(F.col("dst_packets") > 0,
               (F.col("src_packets") / F.col("dst_packets")).cast("double"))
         .otherwise(F.lit(0.0))
    )
)



In [0]:
display(df_features.limit(5))

Hemos decidido conservar los registros en donde hay nulos en community ID y optado por un nuevo id usando uuid

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
import uuid
def get_uuid():
    return str(uuid.uuid4())
uuid_v4=udf(get_uuid,StringType())
df_features=df_features.withColumn("id",uuid_v4())

In [0]:
df_features = df_features.select(
    "id",
    "src_ip",
    "dst_ip",
    "protocol",
    "tot_bytes",
    "tot_packets",
    "tot_data_bytes",
    "session_duration",
    "bytes_per_second",
    "avg_packet_size",
    "bytes_ratio_src_dst",
    "packets_ratio_src_dst"
)

In [0]:
df_features.limit(10).display()

In [0]:
df_features.select([F.count(F.when(F.col(c).isNull(), 1)).alias(c) for c in df_features.columns]).display()

In [0]:
df_features = df_features.dropna()
# Tamaño del dataset
df_features.count()

In [0]:
%sql
use catalog dev;
create schema if not exists feature_store;

In [0]:
%pip install databricks-feature-engineering

In [0]:
%sql
DROP TABLE IF EXISTS dev.ciencias_data.traffic_network_fs

In [0]:
from databricks.feature_engineering  import FeatureEngineeringClient
fe=FeatureEngineeringClient()
fe.create_table(
    name="dev.ciencias_data.traffic_network_fs",
    primary_keys=["id"],
    df=df_features,
    schema=df_features.schema,
    description="features de sesiones de tráfico de red"
)

In [0]:
display(spark.table("dev.ciencias_data.traffic_network_fs").limit(10))
spark.table("dev.ciencias_data.traffic_network_fs").count()

# Inciso 3
###Feature Engineering

Se construyeron nuevas variables a partir de los datos originales con el objetivo de capturar mejor el comportamiento del tráfico de red.
Entre las principales variables generadas se encuentran:

bytes_per_second: mide la velocidad de transmisión de datos por sesión
avg_packet_size: representa el tamaño promedio de los paquetes
bytes_ratio_src_dst: indica la relación entre bytes enviados y recibidos
packets_ratio_src_dst: mide la proporción de paquetes entre origen y destino

Estas variables permiten caracterizar de manera más precisa cada sesión, facilitando la identificación de patrones y anomalías en el tráfico de red.

Finalmente, las features fueron almacenadas en una tabla Delta dentro de Unity Catalog, funcionando como un repositorio centralizado de variables reutilizables para modelos de machine learning.

Ahora haremos un pipeline transformando variables, normalizaremos los datos numericos y haremos onehotencoder para aquellas categorias sin orden

In [0]:
dbutils.library.restartPython()

In [0]:
from databricks.feature_engineering import FeatureLookup
from databricks.feature_engineering import FeatureEngineeringClient
from pyspark.ml.feature import VectorAssembler, MinMaxScaler, StringIndexer, CountVectorizer, OneHotEncoder
import pyspark.sql.functions as F
from pyspark.ml import Pipeline


In [0]:
df_fe = spark.table("dev.ciencias_data.traffic_network_fs")
df_fe.select(
    F.countDistinct("dst_ip").alias("unique_dst_ip"),
    F.countDistinct("src_ip").alias("unique_src_ip"),
    F.countDistinct("protocol").alias("unique_protocol")
).show()

# df_fe.groupBy("protocol").count().orderBy("count", ascending=False).show()

# df_fe.groupBy("dst_ip").count().orderBy("count", ascending=False).show()

# df_fe.groupBy("src_ip").count().orderBy("count", ascending=False).show()



In [0]:
df = spark.table("dev.ciencias_data.traffic_network_fs").select("id")
fe = FeatureEngineeringClient()

def load_data(data, table_name, lookup_key):
    model_feature_lookups = [FeatureLookup(table_name=table_name, lookup_key=lookup_key)]
    training_set = fe.create_training_set(
        df=data,
        feature_lookups=model_feature_lookups,
        label=None
    )
    training_df = training_set.load_df()
    return training_df

# Crear el df_train
df_train = load_data(df, "dev.ciencias_data.traffic_network_fs", "id")
df_train.display()

In [0]:
num_features = [
    'tot_bytes',
    'tot_packets',
    'tot_data_bytes',
    'session_duration',
    'bytes_per_second',
    'avg_packet_size',
    'bytes_ratio_src_dst',
    'packets_ratio_src_dst'
]

vectorizer = CountVectorizer(
    inputCol="protocol", 
    outputCol="protocols_vec", 
    binary=True
)

string_idx = StringIndexer(
    inputCols=["src_ip", "dst_ip"],
    outputCols=["idx_src_ip", "idx_dst_ip"],
    handleInvalid="skip",
    stringOrderType="frequencyDesc" 
)

ohe = OneHotEncoder(
    inputCols=["idx_src_ip", "idx_dst_ip"],
    outputCols=["ohe_src_ip", "ohe_dst_ip"],
    dropLast=False
)

assembler = VectorAssembler(
    inputCols=["protocols_vec", "ohe_src_ip", "ohe_dst_ip"] + num_features, 
    outputCol="features_unscaled",
    handleInvalid="skip" 
)

scaler = MinMaxScaler(
    inputCol="features_unscaled", 
    outputCol="features"
)


In [0]:
from pyspark.ml.clustering import KMeans
kmeans = KMeans(k=3, seed=42, featuresCol="features", predictionCol="cluster")

pipeline_completo = Pipeline(stages=[vectorizer, string_idx, ohe, assembler, scaler, kmeans])

pipeline_model = pipeline_completo.fit(df_train)
predictions = pipeline_model.transform(df_train)

predictions.display()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import FloatType
import numpy as np

kmeans_model = pipeline_model.stages[-1]
centers = kmeans_model.clusterCenters()

def get_distance(features, cluster_idx):
    center = centers[cluster_idx]
    return float(np.linalg.norm(features.toArray() - center))

distance_udf = F.udf(get_distance, FloatType())

df_dist = predictions.withColumn(
    "dist_to_centroid", 
    distance_udf(F.col("features"), F.col("cluster"))
)

threshold = df_dist.approxQuantile("dist_to_centroid", [0.98], 0.01)[0]
print(f"Umbral de anomalía detectado: {threshold}")

df_final = df_dist.withColumn("is_anomaly", F.col("dist_to_centroid") > threshold)

df_final.select(
    "src_ip",
    "dst_ip",
    "protocol", 
    "tot_bytes",
    "tot_packets",
    "tot_data_bytes",
    "session_duration", 
    "cluster", 
    "dist_to_centroid", 
    "is_anomaly"
).show(100, truncate=False)

In [0]:
import matplotlib.pyplot as plt
from pyspark.ml.evaluation import ClusteringEvaluator
import gc 

pipeline_fe = Pipeline(stages=[vectorizer, string_idx, ohe, assembler, scaler])
pipeline_model_fe = pipeline_fe.fit(df_train)
df_features_only = pipeline_model_fe.transform(df_train)


cost = []
k_range = range(2, 8)

print("Calculando Método del Codo...")
for k in k_range:
    km = KMeans(k=k, seed=42, featuresCol="features")
    
    # Entrenar el modelo
    model_km = km.fit(df_features_only) 
    
    # Obtenemos la inercia (costo)
    wcss = model_km.summary.trainingCost
    cost.append(wcss)
    print(f"Para K={k}, el costo WCSS es: {wcss}")
    
    # --- SOLUCIÓN AL ERROR DE MEMORIA ---
    # Eliminamos el modelo de la sesión de Spark Connect
    del model_km
    # Forzamos a Python a limpiar la memoria RAM inmediatamente
    gc.collect() 
    # ------------------------------------

# Graficar el Codo
plt.figure(figsize=(10, 6))
plt.plot(k_range, cost, 'bx-')
plt.xlabel('Número de Clusters (K)')
plt.ylabel('Costo (WCSS)')
plt.title('Método del Codo')
plt.show()